In [ ]:
import geopandas as gpd
import rasterio

# 1. Rutas de tus archivos (Aseguradas con 'r' para evitar conflictos con las '\')
ruta_shapefile = r"C:\Analisis_Geoespacial\2026_I_Analisis_Geoespacial\Data\Puntos\Eventos_y_ausencias.shp"
ruta_raster = r"C:\Analisis_Geoespacial\2026_I_Analisis_Geoespacial\Data\Superficies\DEM\DEM_Antioquia.tif"
nombre_nuevo_campo = "Elev"  # Nombre de la columna que se creará o actualizará

# 2. Cargar el shapefile de puntos original
puntos = gpd.read_file(ruta_shapefile)

# 3. Abrir el raster y extraer los valores
with rasterio.open(ruta_raster) as src:
    # Es CRUCIAL que ambos archivos estén en el mismo Sistema de Referencia de Coordenadas (CRS)
    if puntos.crs != src.crs:
        print(f"Reproyectando puntos de {puntos.crs} a {src.crs}...")
        puntos = puntos.to_crs(src.crs)
    
    # Extraer las coordenadas (X, Y) de cada punto
    coordenadas = [(pt.x, pt.y) for pt in puntos.geometry]
    
    # Muestrear el raster en esas coordenadas
    valores = [val[0] for val in src.sample(coordenadas)]

# 4. Asignar los valores al nuevo campo en el GeoDataFrame
puntos[nombre_nuevo_campo] = valores

# 5. GUARDAR DIRECTAMENTE EN EL ARCHIVO ORIGINAL (Sobrescribir)
puntos.to_file(ruta_shapefile)

print(f"¡Proceso completado con éxito! El campo '{nombre_nuevo_campo}' ha sido agregado al shapefile original: {ruta_shapefile}")

c:\Analisis_Geoespacial\2026_I_Analisis_Geoespacial\.venv\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Value -3.4028234663852886e+38 of field TWI of feature 1 not successfully written. Possibly due to too larger number with respect to field width
  ogr_write(
c:\Analisis_Geoespacial\2026_I_Analisis_Geoespacial\.venv\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Value -3.4028234663852886e+38 of field TWI of feature 90 not successfully written. Possibly due to too larger number with respect to field width
  ogr_write(
c:\Analisis_Geoespacial\2026_I_Analisis_Geoespacial\.venv\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Value -3.4028234663852886e+38 of field TWI of feature 91 not successfully written. Possibly due to too larger number with respect to field width
  ogr_write(
c:\Analisis_Geoespacial\2026_I_Analisis_Geoespacial\.venv\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Value -3.4028234663852886e+38 of field TWI of feature 92 not successfully writt

¡Proceso completado con éxito! El campo 'TWI' ha sido agregado al shapefile original: C:\Analisis_Geoespacial\2026_I_Analisis_Geoespacial\Data\Otros\Shape\Puntos_Snap.shp


c:\Analisis_Geoespacial\2026_I_Analisis_Geoespacial\.venv\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Value -3.4028234663852886e+38 of field TWI of feature 1222 not successfully written. Possibly due to too larger number with respect to field width
  ogr_write(
c:\Analisis_Geoespacial\2026_I_Analisis_Geoespacial\.venv\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Value -3.4028234663852886e+38 of field TWI of feature 1227 not successfully written. Possibly due to too larger number with respect to field width
  ogr_write(


In [ ]:
import geopandas as gpd
import rasterio
import numpy as np
from rasterio.windows import Window

# =====================================================
# RUTAS
# =====================================================
ruta_shapefile = r"C:\Analisis_Geoespacial\2026_I_Analisis_Geoespacial\Data\Puntos\Base_datos_Completa.shp"
ruta_raster = r"C:\Analisis_Geoespacial\2026_I_Analisis_Geoespacial\Data\Superficies\DEM\Curvatura.tif"

nombre_nuevo_campo = "Curv_Mean"

# =====================================================
# CONFIGURACIÓN
# =====================================================
n = 3  # número de celdas alrededor del punto

# Si n = 1, ventana 3x3
# Si n = 2, ventana 5x5
# Si n = 3, ventana 7x7

NODATA_SALIDA = -9999

# =====================================================
# CARGAR SHAPEFILE
# =====================================================
puntos = gpd.read_file(ruta_shapefile)

# =====================================================
# EXTRAER PROMEDIO POR VENTANA
# =====================================================
valores_promedio = []

with rasterio.open(ruta_raster) as src:

    if puntos.crs != src.crs:
        print(f"Reproyectando puntos de {puntos.crs} a {src.crs}...")
        puntos = puntos.to_crs(src.crs)

    nodata = src.nodata

    for pt in puntos.geometry:

        if pt is None or pt.is_empty:
            valores_promedio.append(NODATA_SALIDA)
            continue

        # Convertir coordenadas del punto a fila/columna
        fila, col = src.index(pt.x, pt.y)

        # Definir ventana centrada en el punto
        row_ini = fila - n
        col_ini = col - n
        size = 2 * n + 1

        # Ajustar ventana si cae en los bordes del raster
        row_ini_clip = max(row_ini, 0)
        col_ini_clip = max(col_ini, 0)

        row_fin_clip = min(fila + n + 1, src.height)
        col_fin_clip = min(col + n + 1, src.width)

        height = row_fin_clip - row_ini_clip
        width = col_fin_clip - col_ini_clip

        if height <= 0 or width <= 0:
            valores_promedio.append(NODATA_SALIDA)
            continue

        window = Window(
            col_off=col_ini_clip,
            row_off=row_ini_clip,
            width=width,
            height=height
        )

        data = src.read(1, window=window).astype(float)

        # Excluir NoData
        if nodata is not None:
            data[data == nodata] = np.nan

        data[~np.isfinite(data)] = np.nan

        if np.all(np.isnan(data)):
            valores_promedio.append(NODATA_SALIDA)
        else:
            valores_promedio.append(float(np.nanmean(data)))

# =====================================================
# GUARDAR RESULTADO EN EL SHAPEFILE ORIGINAL
# =====================================================
puntos[nombre_nuevo_campo] = valores_promedio

puntos.to_file(ruta_shapefile)

print("Proceso completado.")
print(f"Campo agregado/actualizado: {nombre_nuevo_campo}")
print(f"Ventana usada: {2*n+1} x {2*n+1}")
print(f"Shapefile actualizado: {ruta_shapefile}")

Leyendo el raster DEM...
⚠ Alerta: El raster está en WGS 84 (grados). Convirtiendo unidades de celda a metros...
Resolución espacial en metros: DX = 30.01m, DY = 29.80m
Calculando pendientes y derivadas secundarias...
Calculando la curvatura de perfil (Método Geométrico Krcho/Evans)...
Guardando el nuevo raster de curvatura...


KeyboardInterrupt: 

In [ ]:
import os
import numpy as np
import rasterio
import geopandas as gpd

from shapely.geometry import Point
from scipy.spatial import cKDTree
from tqdm import tqdm

# =====================================================
# RUTAS DE ENTRADA Y SALIDA
# =====================================================
ruta_raster = r"C:\Analisis_Geoespacial\2026_I_Analisis_Geoespacial\Data\Superficies\DEM\red_raster.tif"
ruta_puntos_inundacion = r"C:\Analisis_Geoespacial\2026_I_Analisis_Geoespacial\Data\Puntos\Base_datos_Completa.shp"
ruta_salida = r"C:\Analisis_Geoespacial\2026_I_Analisis_Geoespacial\Data\Puntos\Ausencias.shp"

# =====================================================
# PARÁMETROS
# =====================================================
numero_puntos = 1000          # cantidad final de puntos aleatorios deseados
distancia_minima = 500        # distancia mínima a puntos de inundación, en unidades del CRS
semilla = 123                 # semilla para reproducibilidad

# =====================================================
# LEER RASTER Y OBTENER CENTROS DE PÍXELES VÁLIDOS
# =====================================================
print("Leyendo raster y detectando píxeles válidos...")

with rasterio.open(ruta_raster) as src:
    raster = src.read(1)
    nodata = src.nodata
    transform = src.transform
    crs_raster = src.crs

    if nodata is not None:
        mascara_valida = raster != nodata
    else:
        mascara_valida = np.isfinite(raster)

    mascara_valida &= np.isfinite(raster)

    filas, cols = np.where(mascara_valida)

    if len(filas) == 0:
        raise ValueError("No se encontraron píxeles válidos en el raster.")

    xs, ys = rasterio.transform.xy(
        transform,
        filas,
        cols,
        offset="center"
    )

    coords_validas = np.column_stack([xs, ys])

print(f"Píxeles válidos encontrados: {len(coords_validas)}")

# =====================================================
# LEER PUNTOS DE INUNDACIÓN
# =====================================================
print("Leyendo puntos de inundación...")

puntos_inundacion = gpd.read_file(ruta_puntos_inundacion)

if puntos_inundacion.crs != crs_raster:
    puntos_inundacion = puntos_inundacion.to_crs(crs_raster)

puntos_inundacion = puntos_inundacion[
    puntos_inundacion.geometry.notnull() &
    (~puntos_inundacion.geometry.is_empty)
].copy()

if len(puntos_inundacion) == 0:
    raise ValueError("El shapefile de inundación no contiene puntos válidos.")

coords_inundacion = np.array([
    (geom.x, geom.y) for geom in puntos_inundacion.geometry
])

# =====================================================
# FILTRAR CANDIDATOS POR DISTANCIA A INUNDACIONES
# =====================================================
print("Filtrando píxeles cercanos a puntos de inundación...")

tree = cKDTree(coords_inundacion)

distancias, _ = tree.query(
    coords_validas,
    k=1,
    workers=-1
)

mascara_filtrada = distancias >= distancia_minima
coords_filtradas = coords_validas[mascara_filtrada]

print(f"Candidatos después del filtro de distancia: {len(coords_filtradas)}")

if len(coords_filtradas) == 0:
    raise ValueError("No quedaron puntos candidatos después del filtro de distancia.")

if len(coords_filtradas) < numero_puntos:
    raise ValueError(
        f"Solo hay {len(coords_filtradas)} candidatos disponibles, "
        f"pero solicitaste {numero_puntos} puntos."
    )

# =====================================================
# SELECCIÓN ALEATORIA
# =====================================================
print("Seleccionando puntos aleatorios...")

rng = np.random.default_rng(semilla)

indices = rng.choice(
    len(coords_filtradas),
    size=numero_puntos,
    replace=False
)

coords_seleccionadas = coords_filtradas[indices]

# =====================================================
# CREAR SHAPEFILE DE SALIDA
# =====================================================
print("Creando shapefile de salida...")

geometria = [
    Point(x, y) for x, y in coords_seleccionadas
]

gdf_salida = gpd.GeoDataFrame(
    {
        "ID": np.arange(1, numero_puntos + 1),
        "Tipo": "No_Inund",
        "Dist_Min": distancias[mascara_filtrada][indices]
    },
    geometry=geometria,
    crs=crs_raster
)

os.makedirs(os.path.dirname(ruta_salida), exist_ok=True)

gdf_salida.to_file(ruta_salida)

print("Proceso finalizado.")
print(f"Puntos generados: {len(gdf_salida)}")
print("Archivo generado:")
print(ruta_salida)

In [ ]:
import geopandas as gpd
from shapely.ops import nearest_points

puntos_path = r"C:\Analisis_Geoespacial\2026_I_Analisis_Geoespacial\Data\Puntos\Base_datos_Completa.shp"
red_path = r"C:\Analisis_Geoespacial\2026_I_Analisis_Geoespacial\Data\Otros\Shape\Red_drenaje.shp"
output_path = r"C:\Analisis_Geoespacial\2026_I_Analisis_Geoespacial\Data\Otros\Shape\Puntos_Snap.shp"

puntos = gpd.read_file(puntos_path)
red = gpd.read_file(red_path)

if puntos.crs != red.crs:
    puntos = puntos.to_crs(red.crs)

red_union = red.geometry.unary_union

def snap_punto(geom, target_geom):
    punto_cercano, vertice_cercano = nearest_points(geom, target_geom)
    return vertice_cercano

puntos_corregidos = puntos.copy()
puntos_corregidos['geometry'] = puntos_corregidos.geometry.apply(lambda p: snap_punto(p, red_union))

puntos_corregidos.to_file(output_path)

print(f"Puntos ajustados guardados exitosamente en: {output_path}")

In [ ]:
import os
import geopandas as gpd
import rasterio
import numpy as np

from rasterio.mask import mask
from tqdm import tqdm

# =====================================================
# RUTAS
# =====================================================
#ruta_shape = r"C:\Analisis_Geoespacial\2026_I_Analisis_Geoespacial\Data\Poligonos\Poligonos\Municipios_Antioquia.shp"
#ruta_shape = r"C:\Analisis_Geoespacial\2026_I_Analisis_Geoespacial\Data\Poligonos\Poligonos\Subzonas_Antioquia.shp"
ruta_shape = r"C:\Analisis_Geoespacial\2026_I_Analisis_Geoespacial\Data\Poligonos\Poligonos\Subzonas_Extension_Antioquia.shp"

#ruta_raster = r"C:\Analisis_Geoespacial\2026_I_Analisis_Geoespacial\Data\Superficies\DEM\Subzonas\DEM.tif"
#ruta_raster = r"C:\Analisis_Geoespacial\2026_I_Analisis_Geoespacial\Data\Superficies\DEM\Subzonas\Pendiente.tif"
#ruta_raster = r"C:\Analisis_Geoespacial\2026_I_Analisis_Geoespacial\Data\Superficies\DEM\Subzonas\Curvatura.tif"
#ruta_raster = r"C:\Analisis_Geoespacial\2026_I_Analisis_Geoespacial\Data\Superficies\DEM\Subzonas\TPI.tif"
#ruta_raster = r"C:\Analisis_Geoespacial\2026_I_Analisis_Geoespacial\Data\Superficies\DEM\Subzonas\TRI.tif"
ruta_raster = r"C:\Analisis_Geoespacial\2026_I_Analisis_Geoespacial\Data\Superficies\LAI\LAI_Subzonas.tif"


# Prefijo de las columnas nuevas
prefijo = "LAI"

# =====================================================
# LEER POLÍGONOS
# =====================================================
gdf = gpd.read_file(ruta_shape)

# =====================================================
# CALCULAR ZONAL STATISTICS
# =====================================================
mean_vals = []
min_vals = []
max_vals = []
std_vals = []  # Nueva lista para Desviación Estándar
cv_vals = []   # Nueva lista para Coeficiente de Variación

with rasterio.open(ruta_raster) as src:

    if gdf.crs != src.crs:
        print(f"Reproyectando shapefile de {gdf.crs} a {src.crs}...")
        gdf = gdf.to_crs(src.crs)

    nodata = src.nodata

    for geom in tqdm(gdf.geometry, total=len(gdf)):

        if geom is None or geom.is_empty:
            mean_vals.append(np.nan)
            min_vals.append(np.nan)
            max_vals.append(np.nan)
            std_vals.append(np.nan)
            cv_vals.append(np.nan)
            continue

        try:
            out_image, out_transform = mask(
                src,
                [geom],
                crop=True,
                filled=True
            )

            data = out_image[0].astype(float)

            if nodata is not None:
                data[data == nodata] = np.nan

            data[~np.isfinite(data)] = np.nan

            if np.all(np.isnan(data)):
                mean_vals.append(np.nan)
                min_vals.append(np.nan)
                max_vals.append(np.nan)
                std_vals.append(np.nan)
                cv_vals.append(np.nan)
            else:
                # Calculamos media y desviación estándar primero para usarlas en el CV
                m_val = float(np.nanmean(data))
                s_val = float(np.nanstd(data))
                
                mean_vals.append(m_val)
                min_vals.append(float(np.nanmin(data)))
                max_vals.append(float(np.nanmax(data)))
                std_vals.append(s_val)
                
                # Evitar división por cero en el Coeficiente de Variación
                if m_val != 0 and not np.isnan(m_val):
                    cv_vals.append(s_val / m_val)
                else:
                    cv_vals.append(np.nan)

        except Exception:
            mean_vals.append(np.nan)
            min_vals.append(np.nan)
            max_vals.append(np.nan)
            std_vals.append(np.nan)
            cv_vals.append(np.nan)

# =====================================================
# AGREGAR / ACTUALIZAR COLUMNAS
# =====================================================
columnas_nuevas = [
    f"{prefijo}_mean", f"{prefijo}_min", f"{prefijo}_max", 
    f"{prefijo}_std", f"{prefijo}_cv"
]

for nombre in columnas_nuevas:
    if nombre in gdf.columns:
        print(f"Actualizando campo: {nombre}")
    else:
        print(f"Creating campo: {nombre}")

gdf[f"{prefijo}_mean"] = mean_vals
gdf[f"{prefijo}_min"] = min_vals
gdf[f"{prefijo}_max"] = max_vals
gdf[f"{prefijo}_std"] = std_vals
gdf[f"{prefijo}_cv"] = cv_vals

# =====================================================
# SOBREESCRIBIR SHAPEFILE
# =====================================================
gdf.to_file(ruta_shape, index=False)

print("Proceso terminado.")
print("Shapefile actualizado correctamente.")

100%|██████████| 48/48 [00:06<00:00,  7.33it/s]

Creando campo: LAI_mean
Creando campo: LAI_min
Creando campo: LAI_max
Proceso terminado.
Shapefile actualizado correctamente.


In [21]:
import os
import glob
import shutil
import tempfile
import geopandas as gpd

# =====================================================
# RUTAS
# =====================================================
#ruta_poligonos = r"C:\Analisis_Geoespacial\2026_I_Analisis_Geoespacial\Data\Poligonos\Poligonos\Municipios_Antioquia.shp"
#ruta_poligonos = r"C:\Analisis_Geoespacial\2026_I_Analisis_Geoespacial\Data\Poligonos\Poligonos\Subzonas_Antioquia.shp"
ruta_poligonos = r"C:\Analisis_Geoespacial\2026_I_Analisis_Geoespacial\Data\Poligonos\Poligonos\Subzonas_Extension_Antioquia.shp"

ruta_puntos = r"C:\Analisis_Geoespacial\2026_I_Analisis_Geoespacial\Data\Puntos\Base_datos_Completa.shp"

# =====================================================
# NOMBRES DE COLUMNAS NUEVAS
# =====================================================
campo_area = "Area"
campo_eventos = "Tot_event"

# =====================================================
# LEER SHAPEFILES
# =====================================================
poligonos = gpd.read_file(ruta_poligonos)
puntos = gpd.read_file(ruta_puntos)

if poligonos.crs is None:
    raise ValueError("El shapefile de polígonos no tiene CRS definido.")

if puntos.crs is None:
    raise ValueError("El shapefile de puntos no tiene CRS definido.")

# =====================================================
# CREAR COPIAS PROYECTADAS PARA CÁLCULOS
# =====================================================
if poligonos.crs.is_geographic:
    crs_area = poligonos.estimate_utm_crs()
    print(f"CRS geográfico detectado. Calculando área en: {crs_area}")
else:
    crs_area = poligonos.crs

poligonos_calc = poligonos.to_crs(crs_area).copy()
puntos_calc = puntos.to_crs(crs_area).copy()

# =====================================================
# CALCULAR ÁREA
# =====================================================
poligonos[campo_area] = poligonos_calc.geometry.area / 1_000_000

# =====================================================
# CONTAR PUNTOS DENTRO DE CADA POLÍGONO
# =====================================================
poligonos_calc = poligonos_calc.reset_index(drop=False).rename(columns={"index": "poly_id"})
poligonos = poligonos.reset_index(drop=True)

join = gpd.sjoin(
    puntos_calc,
    poligonos_calc[["poly_id", "geometry"]],
    how="left",
    predicate="within"
)

conteo = join.groupby("poly_id").size()

poligonos[campo_eventos] = (
    poligonos.index
    .map(conteo)
    .fillna(0)
    .astype(int)
)

# =====================================================
# SOBREESCRIBIR SHAPEFILE ORIGINAL DE FORMA SEGURA
# =====================================================
carpeta_original = os.path.dirname(ruta_poligonos)
nombre_base = os.path.splitext(os.path.basename(ruta_poligonos))[0]

with tempfile.TemporaryDirectory() as tmpdir:
    ruta_temp = os.path.join(tmpdir, nombre_base + ".shp")

    poligonos.to_file(ruta_temp, index=False)

    # Eliminar archivos originales del shapefile
    for archivo in glob.glob(os.path.join(carpeta_original, nombre_base + ".*")):
        os.remove(archivo)

    # Copiar archivos temporales al lugar original
    for archivo in glob.glob(os.path.join(tmpdir, nombre_base + ".*")):
        shutil.copy2(
            archivo,
            os.path.join(carpeta_original, os.path.basename(archivo))
        )

print("Proceso terminado.")
print("Shapefile sobrescrito correctamente:")
print(ruta_poligonos)
print(f"Campo de área agregado/actualizado: {campo_area}")
print(f"Campo de eventos agregado/actualizado: {campo_eventos}")

CRS geográfico detectado. Calculando área en: EPSG:32618
Proceso terminado.
Shapefile sobrescrito correctamente:
C:\Analisis_Geoespacial\2026_I_Analisis_Geoespacial\Data\Poligonos\Poligonos\Subzonas_Extension_Antioquia.shp
Campo de área agregado/actualizado: Area
Campo de eventos agregado/actualizado: Tot_event


In [6]:
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt

from libpysal import weights
from esda import Moran
from mpl_toolkits.axes_grid1 import make_axes_locatable


# =====================================================
# 1. LEER RESULTADOS DEL MODELO ICAR
# =====================================================

ruta = (
    r"C:\Analisis_Geoespacial\2026_I_Analisis_Geoespacial"
    r"\Data\Poligonos\Poligonos\Resultados_ICAR"
    r"\Resultados_ICAR.gpkg"
)
gdf = gpd.read_file(ruta)
gdf.info()


<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 125 entries, 0 to 124
Data columns (total 66 columns):
 #   Column                 Non-Null Count  Dtype   
---  ------                 --------------  -----   
 0   OBJECTID_1             125 non-null    float64 
 1   NOMBRE_ENT             125 non-null    str     
 2   CATEGORIA              125 non-null    str     
 3   DEPARTAMEN             125 non-null    str     
 4   COD_DEPART             125 non-null    str     
 5   COD_MUNICI             125 non-null    str     
 6   AREA_KM                125 non-null    float64 
 7   OBSERVACIO             4 non-null      str     
 8   PK_CUE                 123 non-null    float64 
 9   ACTO_ADMIN             0 non-null      object  
 10  ESCALA_150             0 non-null      float64 
 11  ESCALA_200             0 non-null      float64 
 12  ESCALA_300             0 non-null      float64 
 13  ESCALA_350             0 non-null      float64 
 14  ESCALA_400             0 non-null 